# Лабораторная работа №1: Первичное исследование данных

## 1. Постановка задачи

### Описание датасета
Данный датасет представляет собой исторические котировки акций компаний группы FAANG (Meta, Apple, Amazon, Netflix, Google) дополненные рассчитанными техническими индикаторами (например, скользящие средние, RSI и др.). Данные охватывают ежедневные показатели: цены открытия, закрытия, максимумы, минимумы и объемы торгов.

### Условный заказчик
Аналитический отдел инвестиционного фонда или частный трейдер, разрабатывающий стратегию автоматизированной торговли.

### Возможные задачи ИАД
1. Описательная аналитика: Сравнение волатильности и доходности различных компаний группы FAANG за определенный период.
2. Поиск аномалий: Выявление резких ценовых скачков или падений объемов, не соответствующих рыночному тренду.

## 2. Паспорт датасета

### Загрузка данных

In [ ]:
import pandas as pd
import kagglehub

# 1. Загрузка
path = kagglehub.dataset_download("vishardmehta/faang-stock-market-data-with-technical-indicators")
# Обычно kagglehub скачивает папку. Нам нужно найти внутри CSV.
import os
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# 2. Размер
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")

# 3. Список признаков и типы
print("\nПаспорт столбцов:")
info_df = pd.DataFrame({
    'Тип (pandas)': df.dtypes,
    'Пропуски': df.isnull().sum(),
    'Гипотеза о смысле': [
        "Название компании (тикер)",
        "Дата торгов",
        "Цена открытия",
        "Максимальная цена за день",
        "Минимальная цена за день",
        "Цена закрытия",
        "Объем торгов",
        "Технический индикатор (тренд)",
        # Добавь остальные, если в твоем CSV их больше 8
    ][:len(df.columns)] # Ограничил для примера
})
display(info_df)

# Приведение типов (ВАЖНО по заданию: дата)
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    print("\nТип столбца 'Date' успешно приведен к datetime.")

### Структура данных

In [ ]:
# Информация о столбцах и типах
df.info()

# Статистика по числовым признакам
df.describe()

## 3. Аудит качества данных

### 3.1. Пропуски и дубликаты

In [ ]:
# Пропуски в %
missing_values = df.isnull().mean() * 100
print("Процент пропусков:\n", missing_values[missing_values > 0])

# Дубликаты
duplicates = df.duplicated().sum()
print(f"\nКоличество полных дубликатов: {duplicates}")

### 3.3. Выбросы (пример для одного признака)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Выберем цену закрытия (Close) для анализа выбросов
target_col = 'Close' 

Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[target_col] < lower_bound) | (df[target_col] > upper_bound)]
print(f"Найдено выбросов для {target_col}: {len(outliers)}")

plt.figure(figsize=(10, 5))
sns.boxplot(x=df[target_col])
plt.title(f'Визуализация выбросов для {target_col}')
plt.show()

## 4. Разведочный анализ (EDA)

### 4.1. Распределение числового признака

In [ ]:
# 1. Распределение цены закрытия
plt.figure(figsize=(10, 6))
sns.histplot(df['Close'], kde=True, color='blue')
plt.title('Распределение цен закрытия')
plt.show()

# 2. Зависимость: Цена vs Объем
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Volume', y='Close', hue='ticker' if 'ticker' in df.columns else None)
plt.title('Зависимость цены от объема торгов')
plt.show()

### 4.2. Анализ категориального признака

In [ ]:
# Замените на реальный категориальный признак
cat_col = 'your_categorical_column'

plt.figure(figsize=(10, 6))
top_categories = df[cat_col].value_counts().head(10)
sns.barplot(x=top_categories.values, y=top_categories.index)
plt.title(f'Топ-10 категорий в {cat_col}')
plt.xlabel('Количество')
plt.show()

## 5. Выводы

Детали в файле `report/quality_report.md`